#### **Celda 1: Inspección de Hardware y Ecosistema**

In [1]:
!nvidia-smi
!nvcc --version | tail -n 1

import time
import numpy as np
import cupy as cp

print(cp.cuda.runtime.getDeviceProperties(0)["name"].decode(), "| CuPy", cp.__version__)

Thu Oct  8 05:37:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

**Celda 2: Benchmark con Sincronización Correcta vs Incorrecta**

In [2]:
# Celda 2 - Medición correcta en GPU (asíncrona)
def medir(fn, reps=5, sync=None):
    fn()  # Calentamiento (compila kernels y reserva memoria)
    if sync:
        sync()
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter()
        fn()
        if sync:
            sync()
        ts.append(time.perf_counter() - t0)
    return np.mean(ts), np.std(ts)

sync = cp.cuda.Device().synchronize

print(f"{'n':>5} | {'CPU (s)':>18} | {'GPU (s)':>19} | {'Speedup':>9} | {'GPU sin sync (s)':>20}")
print("-" * 80)

for n in [256, 1024, 4096]:
    A = np.random.rand(n, n).astype(np.float32)
    B = np.random.rand(n, n).astype(np.float32)

    # Transferencia a VRAM fuera de la región medida
    Ag, Bg = cp.asarray(A), cp.asarray(B)

    t_cpu, s_cpu = medir(lambda: A @ B)
    t_gpu, s_gpu = medir(lambda: Ag @ Bg, sync=sync)
    t_mal, _ = medir(lambda: Ag @ Bg) # Sin sincronizar (medición asíncrona errónea)

    speedup = t_cpu / t_gpu
    print(f"{n:5d} | {t_cpu:.4f} ± {s_cpu:.4f} s | {t_gpu:.5f} ± {s_gpu:.5f} s | {speedup:7.1f}x | {t_mal:.6f} s (falso)")

    n |            CPU (s) |             GPU (s) |   Speedup |     GPU sin sync (s)
--------------------------------------------------------------------------------
  256 | 0.0004 ± 0.0000 s | 0.00010 ± 0.00004 s |     3.9x | 0.000033 s (falso)
 1024 | 0.0170 ± 0.0005 s | 0.00090 ± 0.00002 s |    19.0x | 0.000047 s (falso)
 4096 | 1.0923 ± 0.0175 s | 0.03903 ± 0.00739 s |    28.0x | 0.000034 s (falso)
